In [4]:
from pathlib import Path
from typing import Mapping

import dask.array as da
import xarray as xr
from xarray import DataArray, Dataset, DataTree
from spatialdata import SpatialData
from spatialdata.transformations import Scale, Identity
#import sopa
import spatialdata as sd
from napari_spatialdata import Interactive
import napari


#sopa.settings.auto_save_on_disk = False      

def _open_wsi(path: str | Path) -> tuple[str, xr.Dataset, object, dict]:
    import tiffslide

    path = Path(path)
    image_name = path.stem
    slide = tiffslide.open_slide(str(path))
    zarr_store = slide.zarr_group.store
    zarr_img = xr.open_zarr(zarr_store, consolidated=False, mask_and_scale=False)
    print(zarr_img)

    metadata = {
        "properties": slide.properties,
        "dimensions": slide.dimensions,
        "level_count": slide.level_count,
        "level_dimensions": slide.level_dimensions,
        "level_downsamples": slide.level_downsamples,
    }
    return image_name, zarr_img, slide, metadata


def load_cell_dive_spatialdata_tiffslide(
    image_files: Mapping[str, str],  # {channel_name: image_path}
    chunks: tuple[int, int, int] = (1, 1024, 1024),
    coordinate_system: str = "global",
) -> SpatialData:
    """Load Cell DIVE multi-channel multiscale image as SpatialData using tiffslide backend."""

    channel_names = list(image_files.keys())
    paths = list(image_files.values())

    pyramids = []
    base_shape = None
    slide_downsamples = None

    for path in paths:
        _, img, slide, _ = _open_wsi(path)
        levels = [img[str(k)] for k in sorted(img.keys(), key=int)]
        if base_shape is None:
            base_shape = levels[0].shape[-2:]
            slide_downsamples = slide.level_downsamples

        pyramids.append(levels)

    num_levels = len(pyramids[0])
    assert all(len(p) == num_levels for p in pyramids), "All images must have the same number of pyramid levels"

    multiscale_arrays = [da.stack([p[i] for p in pyramids], axis=0) for i in range(num_levels)]

    images = {}
    for level, arr in enumerate(multiscale_arrays):
        data = DataArray(arr, dims=("c", "y", "x")).chunk({"c": chunks[0], "y": chunks[1], "x": chunks[2]})
        scale_y = base_shape[0] / arr.shape[-2]
        scale_x = base_shape[1] / arr.shape[-1]
        transform = (
            Scale([scale_y, scale_x], axes=("y", "x"))
            if (scale_y != 1.0 or scale_x != 1.0)
            else Identity()
        )
        data.coords["c"] = channel_names
        data.attrs["transform"] = {coordinate_system: transform}
        images[f"scale{level}"] = Dataset({"image": data})

    tree = DataTree.from_dict(images)
    return tree



In [5]:
from pathlib import Path

def build_image_dict_from_folder(folder: str | Path) -> dict[str, str]:
    """
    Builds a dictionary mapping 'R<round>_<channel>' to file paths.

    Expected filename format:
        SLIDE-XXXX_<round>.0.X_R000_<dye>_<channel>_...ome.tif[f]
    DAPI may have no <channel>, so handled as a special case.

    Args:
        folder: Directory containing OME-TIFF files.

    Returns:
        Dictionary like {'R1_CD4': '/path/to/file.tiff', 'R1_DAPI': '/path/to/file.tif'}
    """
    folder = Path(folder)
    image_dict = {}

    for file in folder.iterdir():
        if not file.is_file():
            continue
        if not file.name.lower().endswith(".ome.tif"):
            continue

        parts = file.stem.split("_")
        try:
            # Round number from the 2nd part (e.g., 1 from "1.0.4")
            round_str = parts[1]
            round_num = round_str.split(".")[0]

            dye = parts[3]
            channel = parts[4] if len(parts) > 4 and parts[4] else None

            # Special case: DAPI has no channel part
            if dye.upper() == "DAPI" or not channel:
                channel_name = "DAPI"
            else:
                channel_name = channel

            key = f"R{round_num}_{channel_name}"
            image_dict[key] = str(file.resolve())

        except (IndexError, ValueError):
            print(f"⚠️ Skipping file with unrecognized structure: {file.name}")
            continue

    return image_dict

In [7]:
folder_path = r"/mnt/e/SLIDE-0272_preview"
image_dict = build_image_dict_from_folder(folder_path)
image_dict


{'R1_DAPI': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_1.0.4_R000_DAPI__FINAL_F.ome.tif',
 'R10_Arg1-D4E3M-555-nimbus': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555-nimbus_FINAL_AFR_F.ome.tif',
 'R10_Arg1-D4E3M-555': '/mnt/e/SLIDE-0272_preview/SLIDE-0272_10.0.4_R000_Cy3_Arg1-D4E3M-555_FINAL_AFR_F.ome.tif'}

In [8]:
image = load_cell_dive_spatialdata_tiffslide(image_dict, chunks = (1, 512, 512), coordinate_system = "SLIDE")


<xarray.Dataset> Size: 10GB
Dimensions:  (Y: 60899, X: 59212, Y4: 3806, X4: 3700, Y1: 30449, X1: 29606,
              Y6: 951, X6: 925, Y5: 1903, X5: 1850, Y2: 15224, X2: 14803,
              Y3: 7612, X3: 7401)
Dimensions without coordinates: Y, X, Y4, X4, Y1, X1, Y6, X6, Y5, X5, Y2, X2,
                                Y3, X3
Data variables:
    0        (Y, X) uint16 7GB dask.array<chunksize=(512, 512), meta=np.ndarray>
    4        (Y4, X4) uint16 28MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    1        (Y1, X1) uint16 2GB dask.array<chunksize=(512, 512), meta=np.ndarray>
    6        (Y6, X6) uint16 2MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    5        (Y5, X5) uint16 7MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    2        (Y2, X2) uint16 451MB dask.array<chunksize=(512, 512), meta=np.ndarray>
    3        (Y3, X3) uint16 113MB dask.array<chunksize=(512, 512), meta=np.ndarray>
Attributes:
    multiscales:  [{'datasets': [{'path': '0'}, {'path': '1

AssertionError: All images must have the same number of pyramid levels

In [9]:
from spatialdata import SpatialData
import napari

# Load SpatialData object
#sdata = SpatialData.read("test_spatialdata_v3.zarr")

# Access the image DataTree and extract multiscale Dask arrays
#tree = image_0272.images["SLIDE-0275"]
tree = image
multiscale = [tree[f"scale{level}"].image.data for level in range(len(tree.children))]


# Extract channel names from coordinates
channel_names = tree["scale0"].image.coords["c"].values.tolist()


viewer = napari.Viewer()
# Launch napari with multiscale image (channel names only as metadata or printout)
for i, name in enumerate(channel_names):
    channel_data = [level[i] for level in multiscale]  # collect ith channel across scales
    viewer.add_image(
        channel_data,
        multiscale=True,
        name=name,
        scale=(1, 1),  # or pixel size if needed
        blending="additive"
    ) 